# LocalLens: fine-tune the frame-filling model on a Colab GPU

Same recipe as `finetune/train_mlx.py` (which does it on a Mac): 16-bit LoRA, rank 16, alpha 32, all linear layers, loss on the answer only, 2 epochs, effective batch 16, lr 2e-4.
**Runtime > Change runtime type > T4 GPU** (free tier is enough). The data is our own synthetic data (see `finetune/DATA_LICENSES.md`), so uploading it to Colab shares nothing private.

You need two files from your laptop: `finetune/data/train.jsonl` and `finetune/data/val.jsonl` (made by `python -I finetune/build_data.py`).
**Not yet run on Colab**: written against current transformers/peft; if a cell fails, send the error.

In [ ]:
!pip -q install -U transformers peft accelerate datasets safetensors
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: change the runtime type')

## 1. Upload the data

In [ ]:
from google.colab import files
up = files.upload()            # choose train.jsonl and val.jsonl
assert 'train.jsonl' in up and 'val.jsonl' in up

## 2. Tokenise: prompt tokens get label -100, so only the answer is learned

In [ ]:
import json
from transformers import AutoTokenizer
BASE = 'Qwen/Qwen2.5-0.5B-Instruct'            # Apache-2.0
tok = AutoTokenizer.from_pretrained(BASE)
MAXLEN = 2048

def encode(row):
    msgs = row['messages']
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    answer = msgs[-1]['content'] + '<|im_end|>\n'      # the model must learn to stop
    p, a = tok(prompt, add_special_tokens=False)['input_ids'], tok(answer, add_special_tokens=False)['input_ids']
    ids = (p + a)[:MAXLEN]
    labels = ([-100] * len(p) + a)[:MAXLEN]
    return {'input_ids': ids, 'attention_mask': [1] * len(ids), 'labels': labels}

def load(name):
    return [encode(json.loads(l)) for l in open(name) if l.strip()]

train, val = load('train.jsonl'), load('val.jsonl')
print(len(train), len(val), max(len(x['input_ids']) for x in train))

## 3. Train (fp16 base, LoRA adapters in fp32)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16).cuda()
model.gradient_checkpointing_enable(); model.enable_input_require_grads()
cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, task_type='CAUSAL_LM',
                 target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
model = get_peft_model(model, cfg)
for n, p in model.named_parameters():
    if p.requires_grad: p.data = p.data.float()          # adapters in fp32, as the T4 has no bf16
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir='out', per_device_train_batch_size=4, gradient_accumulation_steps=4, num_train_epochs=2,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.05, fp16=True, logging_steps=10,
    eval_strategy='steps', eval_steps=100, save_strategy='no', per_device_eval_batch_size=4,
    report_to='none', seed=1, remove_unused_columns=False)
coll = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=val[:300], data_collator=coll)
trainer.train()

Watch the **eval loss**: it should fall and flatten. If the loss suddenly jumps (the Mac run diverged with too large a multiplier), lower `learning_rate` to 1e-4 and rerun.
The real stopping signal is frame accuracy on the locked test set, which is measured on your laptop after export.

## 4. Merge the adapter into the 16-bit base and download it
Merge into the fp16 base, never into a 4-bit model.

In [ ]:
model = model.merge_and_unload()
model = model.half()
model.save_pretrained('fused', safe_serialization=True); tok.save_pretrained('fused')
!cd fused && ls -la && cd .. && zip -qr fused.zip fused
from google.colab import files; files.download('fused.zip')     # about 1 GB

## 5. On your laptop
Unzip `fused.zip` into `finetune/runs/colab/fused`, then:
```
~/ft-env/bin/python finetune/export_gguf.py --stage f16 --fused finetune/runs/colab/fused --name locallens-colab
.venv/bin/python finetune/export_gguf.py --stage quant --name locallens-colab
.venv/bin/python eval/run_eval.py --set heldout4 --model models/finetuned/locallens-colab-q8_0.gguf --prompt-style compact --show-failures
```